# Fase 4 — Validación de los datos sintéticos de Golazo

**Objetivo:** comprobar que los datos sintéticos generados son **coherentes**: que respetan las cifras reales conocidas del cliente (301.000 vistas y 900 suscriptores en 30 días), que ninguna métrica sale negativa o fuera de rango, y que las distribuciones (demografía, tráfico, retención) tienen sentido antes de pasar al diseño de la base de datos.

**Requisito previo:**
```
python -m src.generador_sintetico
```
Esto genera en `/synthetic`:
- `golazo_catalogo_videos.csv`
- `golazo_analytics_sintetico.json`

In [1]:
import json
import os

import pandas as pd

SYNTHETIC_DIR = os.path.join("..", "synthetic")

df_catalogo = pd.read_csv(os.path.join(SYNTHETIC_DIR, "golazo_catalogo_videos.csv"))

with open(os.path.join(SYNTHETIC_DIR, "golazo_analytics_sintetico.json"), encoding="utf-8") as f:
    analytics = json.load(f)

print(f"Vídeos en el catálogo: {len(df_catalogo)} (real: 338)")

Vídeos en el catálogo: 338 (real: 338)


## 1. Coherencia con las cifras reales conocidas

In [2]:
views_generadas = sum(row[1] for row in analytics["evolucion_diaria"]["rows"])
subs_generados = sum(row[4] for row in analytics["evolucion_diaria"]["rows"])

print(f"Vistas generadas (30 días): {views_generadas:,} | Real: 301,000 | Coincide: {views_generadas == 301000}")
print(f"Suscriptores generados (30 días): {subs_generados} | Real: 900 | Coincide: {subs_generados == 900}")
print(f"Rango de fechas del catálogo: {df_catalogo.fecha_publicacion.min()} -> {df_catalogo.fecha_publicacion.max()}")

Vistas generadas (30 días): 301,000 | Real: 301,000 | Coincide: True
Suscriptores generados (30 días): 900 | Real: 900 | Coincide: True
Rango de fechas del catálogo: 2015-04-24 -> 2026-09-12


## 2. Ningún valor negativo ni fuera de rango

Comprobación defensiva: ninguna métrica de conteo (vistas, suscriptores, likes) debe ser negativa, y los porcentajes deben sumar 100.

In [3]:
# Catálogo de vídeos
assert (df_catalogo[["views_totales", "likes", "comentarios", "duracion_segundos"]] >= 0).all().all()

# Evolución diaria
assert all(row[1] >= 0 and row[4] >= 0 and row[5] >= 0 for row in analytics["evolucion_diaria"]["rows"])

# Tráfico: no negativos y suma exacta
views_trafico = [row[1] for row in analytics["trafico"]["rows"]]
assert all(v >= 0 for v in views_trafico)
assert sum(views_trafico) == 301000

# Demografía: suma ~100
suma_demografia = sum(row[2] for row in analytics["demografia"]["rows"])
assert 99.5 <= suma_demografia <= 100.5

print("Todas las comprobaciones de rango pasaron correctamente.")

Todas las comprobaciones de rango pasaron correctamente.


## 3. Vistazo a las distribuciones

In [4]:
print("Vídeos por categoría de contenido:")
print(df_catalogo["categoria"].value_counts())
print("\nVistas medias por categoría:")
print(df_catalogo.groupby("categoria")["views_totales"].mean().round(0).sort_values(ascending=False))

Vídeos por categoría de contenido:
categoria
Seguimiento del club      154
Opinión post-partido       90
Fichajes                   54
Curiosidades de fútbol     23
Afición                    17
Name: count, dtype: int64

Vistas medias por categoría:
categoria
Afición                   413.0
Curiosidades de fútbol    254.0
Opinión post-partido      254.0
Fichajes                  253.0
Seguimiento del club      246.0
Name: views_totales, dtype: float64


In [5]:
df_trafico = pd.DataFrame(
    analytics["trafico"]["rows"],
    columns=[c["name"] for c in analytics["trafico"]["columnHeaders"]],
)
df_trafico["pct"] = (df_trafico["views"] / df_trafico["views"].sum() * 100).round(1)
df_trafico.sort_values("views", ascending=False)

,insightTrafficSourceType,views,estimatedMinutesWatched,pct
0,SUGGESTED_VIDEO,118191,524884,39.3
1,YT_SEARCH,73779,242804,24.5
2,BROWSE_FEATURES,42079,152362,14.0
3,NOTIFICATION,27741,136335,9.2
4,EXTERNAL,19780,117319,6.6
5,PLAYLIST,13117,57986,4.4
6,SHORTS,6313,21127,2.1


In [6]:
df_retencion = pd.DataFrame(
    analytics["retencion_audiencia"]["rows"],
    columns=[c["name"] for c in analytics["retencion_audiencia"]["columnHeaders"]],
)

# Curva de retención media entre todos los vídeos de la muestra
curva_media = df_retencion.groupby("elapsedVideoTimeRatio")["audienceWatchRatio"].mean()
curva_media

elapsedVideoTimeRatio
0.00    0.994587
0.05    0.536320
0.10    0.493453
0.15    0.466773
0.20    0.453347
0.25    0.427940
0.30    0.411320
0.35    0.389200
0.40    0.373113
0.45    0.359967
0.50    0.339360
0.55    0.327667
0.60    0.312433
0.65    0.301987
0.70    0.277827
0.75    0.273793
0.80    0.259520
0.85    0.248173
0.90    0.241427
0.95    0.254180
1.00    0.242773
Name: audienceWatchRatio, dtype: float64

## Conclusión

Los datos sintéticos respetan las cifras reales conocidas del cliente, no contienen valores negativos ni distribuciones que sumen fuera de rango, y siguen patrones de forma (duración, ratios de engagement) heredados de datos reales de @PuroBalompie. Con el catálogo de vídeos y los 5 informes de Analytics ya generados y validados, el siguiente paso es diseñar el modelo entidad-relación de la base de datos (Fase 5) a partir de las entidades ya identificadas: canal, vídeo, evolución diaria, retención, demografía, tráfico e ingresos.